# Lab 06 — SMS Spam Classifier (Naive Bayes, no sklearn)
**Probability Meets Text Track** · Intermediate · ~60 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Tokenise SMS text and remove stopwords
2. Train multinomial Naive Bayes with Laplace smoothing from scratch
3. Evaluate on a seed=42 holdout with a confusion matrix
4. Rank spam-indicative tokens by log-likelihood ratio

## Datasets (this folder)
- `sms.tsv` — auto-download from `https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-01-foundations-python-math-data/labs/lab-06-naive-bayes-sms-spam/lab-06-naive-bayes-sms-spam.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`sms.tsv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-06-naive-bayes-sms-spam"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-01-foundations-python-math-data/labs/lab-06-naive-bayes-sms-spam/bundle/dataset.zip"
NEED = ["sms.tsv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Probability Meets Text Track: Multinomial NB from Scratch

> **Scenario:** `sms.tsv` has ~5572 labeled SMS messages (ham/spam). Build a **multinomial Naive Bayes** classifier with Laplace smoothing using only the Python standard library — then evaluate on a 20% holdout with `seed=42`.
>
> **You will learn:** tokenisation, stopwords, word counts, log-priors, Bayes rule, train/test split, confusion matrix by hand.
> **Time:** ~60 minutes. **Level:** Intermediate. **Needs:** Python 3.8+ only. **Env:** 🟢 Colab only.

The decision this lab supports is operational: *can a transparent, hand-built probability model flag SMS spam well enough to trust its errors?* You will not call `sklearn` — every prior, count, and log-likelihood is written out so you can see exactly where the numbers come from and where the “naive” assumption enters. The end artifact is not just an accuracy score but a full confusion matrix on a fixed seed=42 holdout, plus a ranked list of the tokens that most separate spam from ham. Those three views (score, errors, evidence) are what you would bring to anyone asking whether the classifier is safe to deploy or where it still fails.

### Naive Bayes mental map

| Concept | Code shape |
|---|---|
| Prior P(label) | `count(label) / N` in log space |
| Likelihood P(word\|label) | `(count[word] + α) / (total + α·V)` |
| Score a message | `log prior + Σ log likelihood(token)` |
| Predict | `argmax` over the two labels |
| Holdout | `random.seed(42)` + `shuffle` + 80/20 split |

The table is the whole algorithm in five rows. Start from Bayes' rule: to choose the best label you compare P(label) × ∏ P(wordᵢ | label) across labels — but multiplying many small probabilities underflows floats, so the lab works entirely in **log space**, where the product becomes a sum (`log prior + Σ log likelihood`). The likelihood row is the multinomial formula with **Laplace smoothing** (α): add α to every word count and α·V to the denominator so a word you never saw in training still gets a tiny, nonzero probability instead of a catastrophic zero. `argmax` over the two summed scores gives the prediction, and the holdout row guarantees the evaluation uses messages the model has never counted. Sections 1–6 build each row of this table in order, then the exercises probe where the model is confident and where “free” trips it up.

Two design choices deserve emphasis up front. First, **why no sklearn**: the goal is to see every quantity — priors, smoothed likelihoods, the log-sum — that a library would hide behind one `fit()` call; once you can read those, `MultinomialNB` becomes a thin wrapper you can trust or debug. Second, **why log space**: spam messages contain dozens of tokens, each contributing a likelihood well below 1; their raw product underflows toward 0.0 in IEEE floats and loses all discrimination. Adding logs keeps every term finite and turns the comparison into simple addition, which is also how the implementation below is written.

---

### 1. Load data (local first, Colab fallback)

**Why:** Before building probabilities you need a clean (label, text) list and the base rates of ham vs spam — those base rates become your log-priors later. This step also validates the TSV: only rows whose first field is exactly `ham` or `spam` are kept, so a malformed line can never poison a count. Knowing 4825 vs 747 up front tells you the classes are imbalanced, which is why accuracy alone will not be the final word.

In [ ]:
import csv, math, os, random, re
from collections import Counter

def load_sms():
    local = "sms.tsv"
    if not os.path.exists(local):
        import urllib.request
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv",
            local,
        )
    rows = []
    with open(local, newline="", encoding="utf-8") as f:
        for r in csv.reader(f, delimiter="\t"):
            if len(r) >= 2 and r[0] in ("ham", "spam"):
                rows.append((r[0], r[1]))
    return rows

msgs = load_sms()
print(len(msgs), "messages")
print(Counter(l for l, _ in msgs))
# 5572 messages  Counter({'ham': 4825, 'spam': 747})
print(msgs[0])


**What to notice:**
- **5572** messages loaded — matches the documented size of `sms.tsv`.
- Class balance is skewed: **ham 4825** vs **spam 747** (spam ≈ 13.4% of rows) — a majority-ham classifier would already look “okay” on accuracy, so later metrics must look at the confusion matrix, not accuracy alone.
- Each row is a `(label, text)` tuple; the first row printed shows the real format you will tokenize next.

> **Pitfall:** If your count is not 5572, check encoding and the `r[0] in ("ham", "spam")` filter — a stray header or quoted label row changes every downstream prior and the seed=42 split will not match the expected confusion counts.

---

### 2. Tokenise

**Why:** Naive Bayes never sees characters or syntax — only discrete tokens — so your tokenizer *is* your feature space. Lowercasing and the `[a-z0-9']+` pattern collapse `FREE`/`free` and keep contractions like `don't` intact; dropping single characters and a small stopword list removes ultra-common glue words that carry little label information and only dilute the likelihood ratios. Getting tokenisation right here is what makes the later log-lift ranking interpretable.

In [ ]:
STOP = set(
    "a an the and or but if in on at to for of is are was were be been "
    "i you he she it we they my your our their this that with as so not "
    "no do does did have has had will would can could".split()
)

def tokens(text):
    return [w for w in re.findall(r"[a-z0-9']+", text.lower())
            if w not in STOP and len(w) > 1]

print(tokens("FREE entry in 2 a weekly comp to win FA cup final tickets!"))
# ['free', 'entry', '2', 'weekly', 'comp', 'win', 'fa', 'cup', 'final', 'tickets']


**What to notice:**
- `FREE` became `free` — case folding merges spelling variants so their counts combine.
- Stopwords vanished: `in`, `a`, `to` are gone; informative content words (`free`, `win`, `tickets`) remain.
- `2` survived because `len(w) > 1` keeps multi-character numerals — price-like digits (`150p`, `1000`) matter for spam.
- The example keeps `fa` and `comp`, abbreviations that are short but still longer than one character.

> **Pitfall:** Stopword lists are English-centric and blunt. Removing `no`/`not` can erase negation (“not spammy phrasing” flips meaning), and domain words that look like stopwords may actually carry signal. Treat the list as a tunable choice, freeze it before train/test, and remember every tokenization decision must be identical at train and scoring time — or your likelihoods will silently mismatch.

---

### 3. Train / test split (seed = 42)

**Why:** The confusion matrix and accuracy you report must come from messages the model never counted, or you are just testing memorization. `random.seed(42)` before `shuffle` makes that holdout reproducible across machines and Colab restarts — every expected count later (0.9803 accuracy, 962/15/7/130 cells) depends on this exact split. Shuffling first also keeps ham/spam proportions roughly stable in both slices without a hand-rolled stratifier.

In [ ]:
random.seed(42)
shuffled = msgs[:]
random.shuffle(shuffled)
n_test = int(len(shuffled) * 0.2)
test, train = shuffled[:n_test], shuffled[n_test:]
print(len(train), len(test))   # 4458 1114
print("train labels:", Counter(l for l, _ in train))
# train labels: Counter({'ham': 3848, 'spam': 610})


**What to notice:**
- Split sizes are **4458 train / 1114 test** — `int(5572 * 0.2) = 1114`, with the remainder in train.
- Train labels: **ham 3848, spam 610** — class mix is preserved in spirit (spam still a minority), so priors remain realistic.
- `seed(42)` + `shuffle` on a **copy** (`msgs[:]`) leaves the original list untouched for any later re-run.
- The slice order matters: `test` is the first 20% *after* shuffle; changing seed, shuffle order, or train/test order changes every holdout statistic below.

> **Pitfall:** Setting the seed *after* the shuffle, or shuffling `msgs` in place and reloading differently, produces a different holdout and your confusion counts will not match the expected 962/15/7/130. Seed first, shuffle once, slice once — and never fit vocabulary or counts on `test`.

---

### 4. Fit multinomial NB with Laplace (α = 1)

**Why:** This is the model itself: turn training labels into a log-prior per class and a smoothed P(word | label) for every token, then score new messages by Bayes' rule in log space. **Laplace smoothing** (α = 1) adds one pseudo-count to every (label, word) cell — without it, any token absent from a class's training counts gets likelihood 0, the whole log-sum becomes −∞, and a single unseen word could never be outweighed by strong evidence for the other class. Adding α·V to the denominator keeps each class's likelihoods summing sensibly over the vocabulary while giving rare and out-of-vocabulary (OOV) tokens a small, fair probability instead of a hard veto. **Log-space accumulation** matters for the same numerical reason: a 10-token message multiplies ~10 likelihoods each below 1; in raw probability that product can underflow to 0.0, but the sum of logs stays finite and preserves the ranking. That is why `prior_log`, `log_likelihood`, and `score += ...` never multiply raw probabilities anywhere below.

Why not `sklearn.naive_bayes.MultinomialNB`? Because the learning goal is the *mechanics*: where priors come from, how α enters numerator and denominator, how OOV tokens are handled, and why argmax in log space equals argmax in probability space. A library call would compress all of that into one line — correct, but invisible. Once you can read the from-scratch version, the sklearn comparison in “What to learn next” becomes a verification step rather than a black box.

In [ ]:
ALPHA = 1.0
label_counts = Counter(l for l, _ in train)
N_train = len(train)
prior_log = {lab: math.log(label_counts[lab] / N_train) for lab in label_counts}
# ham ≈ -0.1471, spam ≈ -1.989

word_counts = {lab: Counter() for lab in label_counts}
vocab = set()
for lab, text in train:
    for w in tokens(text):
        word_counts[lab][w] += 1
        vocab.add(w)
V = len(vocab)
# vocab_size ≈ 7826

totals = {lab: sum(word_counts[lab].values()) + ALPHA * V for lab in label_counts}

def log_likelihood(lab, w):
    return math.log((word_counts[lab][w] + ALPHA) / totals[lab])

def classify(text):
    best, best_score = None, -math.inf
    for lab in label_counts:
        score = prior_log[lab]
        for w in tokens(text):
            if w in vocab:
                score += log_likelihood(lab, w)
            else:
                # OOV token: same α / total for every label → constant offset;
                # still include for correctness of the sum
                score += math.log(ALPHA / totals[lab])
        if score > best_score:
            best, best_score = lab, score
    return best

print(classify("Congratulations! You've won a free prize. Claim now!"))  # spam
print(classify("Hey, are we still on for lunch at noon?"))               # ham


**What to notice:**
- Log-priors: **ham ≈ −0.1471**, **spam ≈ −1.989** — spam starts roughly 1.8 nats behind because it is the minority class (~610 / 4458 in train); spam messages must earn that deficit back with spammy tokens.
- Vocabulary size **V ≈ 7826** distinct tokens after stopword removal — every one of them receives +α in each class thanks to smoothing.
- `totals[lab]` already includes `ALPHA * V`, so denominators match the numerator's pseudo-counts (the classic MultinomialNB bookkeeping).
- The prize/congrats message scores as **spam**; the lunch question scores as **ham** — priors plus token evidence are doing sensible work on one-line probes.
- OOV branch: words never seen in training still contribute `log(ALPHA / totals[lab])` — a small negative, slightly different per class because totals differ, never −∞.

> **Pitfall:** α is a bias knob, not a free lunch. α = 0 reproduces the zero-likelihood bug; huge α washes out real word differences toward uniform. α = 1 is the standard default, but if you tune α on the holdout you leak test information — pick it up front or via a train-only validation split. Also remember the “naive” step: tokens are treated as conditionally independent given the label, so repeated correlated words (`free free free`) are multiplied as if they were new evidence.

---

### 5. Evaluate on holdout

**Why:** Accuracy alone would hide the errors that matter — in spam filtering, shipping false positives (real SMS marked spam) is often worse than missing a spam. Scoring the fixed seed=42 holdout and building the 2×2 confusion matrix by hand shows exactly where the 22 mistakes fall: 15 ham→spam false alarms vs 7 spam→ham misses. Those cells, not the headline accuracy, drive threshold and cost discussions with stakeholders.

In [ ]:
preds = [classify(t) for _, t in test]
gold  = [l for l, _ in test]
acc = sum(p == g for p, g in zip(preds, gold)) / len(gold)
print(f"accuracy: {acc:.4f}")  # 0.9803
print("correct:", sum(p == g for p, g in zip(preds, gold)), "/", len(gold))  # 1092 / 1114

conf = Counter(zip(gold, preds))
print("ham->ham", conf[("ham", "ham")], " ham->spam", conf[("ham", "spam")])
print("spam->ham", conf[("spam", "ham")], " spam->spam", conf[("spam", "spam")])
# ham->ham 962  ham->spam 15
# spam->ham 7  spam->spam 130


Confusion matrix layout (rows = true, cols = predicted):

|  | pred ham | pred spam |
|---|---|---|
| **true ham** | 962 | 15 |
| **true spam** | 7 | 130 |

**What to notice:**
- Accuracy **0.9803** — **1092 / 1114** holdout messages correct on data never seen during training.
- True ham mostly stays ham (**962**), but **15** legitimate messages are misfiled as spam (false positives — the costly error class for SMS).
- True spam is caught **130 / 137**; only **7** spam messages slip through as ham (false negatives).
- Row + column totals reconstruct the holdout: 962+15 = 977 ham, 7+130 = 137 spam, total 1114 — a quick integrity check on the matrix.
- Spam recall here is 130/137 ≈ **0.9489** and ham precision is 962/977 ≈ **0.9928** (both appear again as follow-up checks).

> **Pitfall:** Do not quote 0.9803 without the matrix and the base rates. With ~87% ham in the holdout, a trivial “always predict ham” baseline would already score ~0.87 — the 0.98 is impressive because spam is actually detected, not because accuracy is high. Always report at least one spam-class metric alongside accuracy.

---

### 6. Top spam-indicative tokens (log lift)

**Why:** A classifier you cannot interrogate is hard to defend to a stakeholder. Ranking tokens by log P(w|spam) − log P(w|ham) shows *which* words move the decision boundary and by how much — the probabilistic analogue of “show me the features.” The support filter (s + h ≥ 10) keeps one-off coincidences from dominating, so the list reflects words with enough evidence to trust, and the same quantity is exactly what Exercise 1 asks you to reproduce.

In [ ]:
spam_lift = []
for w in vocab:
    ps = (word_counts["spam"][w] + ALPHA) / totals["spam"]
    ph = (word_counts["ham"][w] + ALPHA) / totals["ham"]
    spam_lift.append((w, math.log(ps / ph),
                      word_counts["spam"][w], word_counts["ham"][w]))
spam_lift.sort(key=lambda x: -x[1])
top = [(w, round(lift, 3), s, h) for w, lift, s, h in spam_lift if s + h >= 10][:10]
for row in top:
    print(row)
# claim 5.343 (90 spam / 0 ham in train)
# prize 5.109
# won 4.959
# 150p 4.875
# tone 4.724
# ...


**What to notice:**
- Top lifts: **claim ≈ 5.343** (90 spam / 0 ham in train), **prize ≈ 5.109**, **won ≈ 4.959**, **150p ≈ 4.875**, **tone ≈ 4.724** — marketing and payment jargon, exactly the campaign vocabulary you would expect.
- A log-lift of ~5 means P(w|spam) is roughly e⁵ ≈ 150× P(w|ham) after smoothing — enormous single-token evidence toward spam.
- The `s + h >= 10` filter discards tokens that look magical only because they appeared once or twice.
- Ham-favoring words would appear at the *bottom* of the sorted list (negative log-lift); the head of the list is pure spam signal.

> **Pitfall:** High lift ≠ causal importance, and lift ignores position, phrases, and sender context. A token like `free` can also appear in genuine ham (see Exercise 3) — single-token evidence is strong but not decisive, which is why the model still makes 15 false-positive calls on the holdout.

---

## Exercises (do these!)

### Exercise 1 — Top-10 spam-indicative tokens
Using training counts only, rank tokens by `log P(w|spam) − log P(w|ham)` (support ≥ 10). Print the top 10.
*Expected (approx): claim, prize, won, 150p, tone, 18, www, guaranteed, 1000, 500.*

**Follow-up:** What is spam recall on the holdout? Check: about 0.9489.

<details>
<summary>Hint</summary>

Same computation as Section 6; filter `s + h >= 10` before sorting.
</details>

### Exercise 2 — Accuracy on 20% holdout (seed 42)
Report accuracy to 4 d.p. and the 4 confusion counts.
*Expected: accuracy ≈ 0.9803 · TP_spam=130, FN_spam=7, FP_spam=15, TN=962.*

**Follow-up:** What is ham precision? Check: about 0.9928.

<details>
<summary>Hint</summary>

`random.seed(42); random.shuffle(...)` **before** slicing — order of operations matters.
</details>

### Exercise 3 — Why “free” alone misclassifies marketing ham
How many holdout messages contain `"free"`? How many of those are **ham**? Give one example of a ham message with “free” that the bag-of-words model might over-weight.
*Expected: 48 test messages contain “free”; 14 are ham (e.g. “When you get free, call me”). Naive Bayes multiplies independent token evidence — a rare spammy co-occurrence pattern can tip borderline ham over the decision boundary.*

**Follow-up:** Compare P free given spam vs P free given ham in test — why is free so spammy? Check: about 0.2482 vs 0.0143.

<details>
<summary>Hint</summary>

```python
free_test = [(t, g) for (g, t) in test if "free" in t.lower()]
print(len(free_test), sum(1 for _, g in free_test if g == "ham"))
```

Single tokens cannot capture context; that’s the “naive” independence assumption.
</details>

---

## Solutions

In [ ]:
# --- Solution 1 ---
lift = []
for w in vocab:
    ps = (word_counts["spam"][w] + ALPHA) / totals["spam"]
    ph = (word_counts["ham"][w] + ALPHA) / totals["ham"]
    s = word_counts["spam"][w]; h = word_counts["ham"][w]
    if s + h >= 10:
        lift.append((w, math.log(ps/ph), s, h))
lift.sort(key=lambda x: -x[1])
print([w for w, *_ in lift[:10]])
# ['claim', 'prize', 'won', '150p', 'tone', '18', 'www', 'guaranteed', '1000', '500']

# --- Solution 2 ---
preds = [classify(t) for _, t in test]
gold  = [l for l, _ in test]
acc = sum(p == g for p, g in zip(preds, gold)) / len(gold)
print(f"{acc:.4f}")  # 0.9803
print(Counter(zip(gold, preds)))
# 962 ham->ham, 15 ham->spam, 7 spam->ham, 130 spam->spam

# --- Solution 3 ---
free_test = [(t, g) for (g, t) in test if "free" in t.lower()]
n_free = len(free_test)
n_ham = sum(1 for _, g in free_test if g == "ham")
example = next(t for t, g in free_test if g == "ham")
print(n_free, n_ham)
print(example[:80])
# 48 14
# e.g. "When you get free, call me" / "Now am free call me pa."

# --- Follow-up 1 ---
rec = sum(1 for g, p in zip(gold, preds) if g == "spam" and p == "spam") / sum(1 for g in gold if g == "spam")
rec = round(rec, 4)
print(rec)  # ~0.9489
assert rec == 0.9489

# --- Follow-up 2 ---
hp = sum(1 for g, p in zip(gold, preds) if g == "ham" and p == "ham") / sum(1 for p in preds if p == "ham")
hp = round(hp, 4)
print(hp)  # ~0.9928
assert hp == 0.9928

# --- Follow-up 3 ---
n_spam = sum(1 for g in gold if g == "spam")
n_ham = sum(1 for g in gold if g == "ham")
p_fs = round(sum(1 for _, g in free_test if g == "spam") / n_spam, 4)
p_fh = round(n_ham and sum(1 for _, g in free_test if g == "ham") / n_ham, 4)
print(p_fs, p_fh)  # ~0.2482 0.0143
assert p_fs == 0.2482 and p_fh == 0.0143


### What to learn next
- Add bigrams (`"free entry"`) — reduces “free” false positives.
- Compare to `sklearn.naive_bayes.MultinomialNB` (should land ≈ same accuracy).
- TF-IDF weights; threshold tuning for precision vs recall on spam.
- Cheat sheet: tokenize → count → log-priors + log-likelihoods → argmax → confusion matrix.

*Files in this folder: `sms.tsv`. Paste any block into Python/Jupyter and run top-to-bottom.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
